[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module2/05-testing-pytest.ipynb)

# Testing with pytest
**Module 2 — Intermediate Python | Estimated time: 30 minutes**

## Learning Objectives
- Write tests with **`unittest.TestCase`** including `setUp`/`tearDown` and assertion methods
- Use **`pytest`** — simpler syntax, plain `assert`, and automatic test discovery
- Catch expected exceptions with **`pytest.raises`**
- Share test state with **`@pytest.fixture`**
- Run the same test with multiple inputs using **`@pytest.mark.parametrize`**
- Isolate code under test with **`unittest.mock.MagicMock`** and **`patch`**
- Test a small class end-to-end with fixtures and mocking

In [ ]:
!pip install pytest pytest-cov --quiet
print('pytest installed.')

## 1. The Code Under Test

We need something to test. Let's define a small `BankAccount` class and write it to a file so pytest can discover the tests.

In [ ]:
%%writefile /tmp/bank_account.py
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import datetime


class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the available balance."""


@dataclass
class Transaction:
    amount: float
    kind: str          # 'deposit' or 'withdrawal'
    timestamp: datetime.datetime = field(default_factory=datetime.datetime.utcnow)


class BankAccount:
    """Simple bank account with deposit, withdrawal, and history."""

    def __init__(self, owner: str, initial_balance: float = 0.0):
        if initial_balance < 0:
            raise ValueError('Initial balance cannot be negative')
        self.owner = owner
        self._balance = initial_balance
        self._history: list[Transaction] = []

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> float:
        if amount <= 0:
            raise ValueError(f'Deposit amount must be positive, got {amount}')
        self._balance += amount
        self._history.append(Transaction(amount, 'deposit'))
        return self._balance

    def withdraw(self, amount: float) -> float:
        if amount <= 0:
            raise ValueError(f'Withdrawal amount must be positive, got {amount}')
        if amount > self._balance:
            raise InsufficientFundsError(
                f'Cannot withdraw {amount:.2f}; balance is {self._balance:.2f}'
            )
        self._balance -= amount
        self._history.append(Transaction(amount, 'withdrawal'))
        return self._balance

    def transfer_to(self, target: BankAccount, amount: float) -> None:
        self.withdraw(amount)    # raises InsufficientFundsError if needed
        target.deposit(amount)

    def statement(self) -> list[str]:
        lines = [f'Account: {self.owner}  Balance: ${self._balance:.2f}']
        for tx in self._history:
            sign = '+' if tx.kind == 'deposit' else '-'
            lines.append(f'  {sign}${tx.amount:.2f}  ({tx.kind})')
        return lines

print('bank_account.py written.')

## 2. `unittest.TestCase` Basics

`unittest` is in the standard library. Tests live in a class that inherits `TestCase`; test methods must start with `test_`.

In [ ]:
%%writefile /tmp/test_bank_unittest.py
import sys
sys.path.insert(0, '/tmp')
import unittest
from bank_account import BankAccount, InsufficientFundsError


class TestBankAccountUnittest(unittest.TestCase):

    def setUp(self):
        """Called before every test method."""
        self.account = BankAccount('Alice', initial_balance=100.0)

    def tearDown(self):
        """Called after every test method (for cleanup)."""
        pass   # nothing to clean up in this case

    def test_initial_balance(self):
        self.assertEqual(self.account.balance, 100.0)

    def test_deposit_increases_balance(self):
        self.account.deposit(50.0)
        self.assertEqual(self.account.balance, 150.0)

    def test_withdrawal_decreases_balance(self):
        self.account.withdraw(30.0)
        self.assertAlmostEqual(self.account.balance, 70.0)

    def test_insufficient_funds_raises(self):
        with self.assertRaises(InsufficientFundsError):
            self.account.withdraw(999.0)

    def test_negative_deposit_raises(self):
        with self.assertRaises(ValueError):
            self.account.deposit(-10.0)

    def test_statement_contains_owner(self):
        lines = self.account.statement()
        self.assertTrue(any('Alice' in line for line in lines))


if __name__ == '__main__':
    unittest.main(verbosity=2)

print('test_bank_unittest.py written.')

In [ ]:
!python -m pytest /tmp/test_bank_unittest.py -v 2>&1

## 3. pytest — Simpler Syntax

pytest finds test files automatically (`test_*.py`), uses plain `assert` statements (no special assertion methods needed), and produces readable failure messages.

In [ ]:
%%writefile /tmp/test_bank_pytest.py
import sys
sys.path.insert(0, '/tmp')
import pytest
from bank_account import BankAccount, InsufficientFundsError


# --- Basic pytest tests (no class needed) ---

def test_deposit():
    acc = BankAccount('Bob', 0.0)
    acc.deposit(200.0)
    assert acc.balance == 200.0


def test_overdraft_error_message():
    acc = BankAccount('Carol', 50.0)
    with pytest.raises(InsufficientFundsError, match='Cannot withdraw 100.00'):
        acc.withdraw(100.0)


def test_transfer_moves_funds():
    src = BankAccount('Dave', 500.0)
    dst = BankAccount('Eve', 100.0)
    src.transfer_to(dst, 200.0)
    assert src.balance == 300.0
    assert dst.balance == 300.0


def test_transfer_raises_on_overdraft():
    src = BankAccount('Frank', 10.0)
    dst = BankAccount('Grace', 0.0)
    with pytest.raises(InsufficientFundsError):
        src.transfer_to(dst, 50.0)
    # Ensure balances are unchanged after failed transfer
    assert src.balance == 10.0
    assert dst.balance == 0.0

print('test_bank_pytest.py written.')

In [ ]:
!python -m pytest /tmp/test_bank_pytest.py -v 2>&1

## 4. Fixtures — Shared Test State

`@pytest.fixture` creates a reusable setup function. pytest injects it by matching the parameter name in your test functions.

In [ ]:
%%writefile /tmp/test_fixtures.py
import sys
sys.path.insert(0, '/tmp')
import pytest
from bank_account import BankAccount, InsufficientFundsError


@pytest.fixture
def empty_account():
    """A brand-new account with zero balance."""
    return BankAccount('TestUser', 0.0)


@pytest.fixture
def funded_account():
    """An account pre-loaded with 1000 for testing withdrawals."""
    acc = BankAccount('WealthyUser', 1000.0)
    acc.deposit(500.0)    # history has one transaction
    return acc


@pytest.fixture
def account_pair(funded_account):
    """A tuple (source, destination) for transfer tests.  Fixtures can use other fixtures."""
    dst = BankAccount('Recipient', 0.0)
    return funded_account, dst


def test_empty_balance(empty_account):
    assert empty_account.balance == 0.0


def test_funded_account_initial(funded_account):
    assert funded_account.balance == 1500.0
    assert len(funded_account._history) == 1


def test_transfer_pair(account_pair):
    src, dst = account_pair
    src.transfer_to(dst, 600.0)
    assert src.balance == 900.0
    assert dst.balance == 600.0


def test_statement_shows_transactions(funded_account):
    funded_account.withdraw(200.0)
    lines = funded_account.statement()
    assert any('+$500.00' in line for line in lines)
    assert any('-$200.00' in line for line in lines)

print('test_fixtures.py written.')

In [ ]:
!python -m pytest /tmp/test_fixtures.py -v 2>&1

## 5. `@pytest.mark.parametrize` — Data-Driven Tests

Run the same test logic with multiple input/expected pairs without copy-pasting test functions.

In [ ]:
%%writefile /tmp/test_parametrize.py
import sys
sys.path.insert(0, '/tmp')
import pytest
from bank_account import BankAccount, InsufficientFundsError


@pytest.mark.parametrize('initial, deposit_amt, expected', [
    (0.0,   100.0, 100.0),
    (50.0,  50.0,  100.0),
    (999.0, 1.0,   1000.0),
    (0.0,   0.01,  0.01),
])
def test_deposit_parametrized(initial, deposit_amt, expected):
    acc = BankAccount('x', initial)
    acc.deposit(deposit_amt)
    assert pytest.approx(acc.balance) == expected


@pytest.mark.parametrize('amount, error_type', [
    (-50.0,  ValueError),
    (0.0,    ValueError),
    (10000.0, InsufficientFundsError),
])
def test_withdraw_invalid(amount, error_type):
    acc = BankAccount('x', 100.0)
    with pytest.raises(error_type):
        acc.withdraw(amount)


# Parametrize with ids for clearer output
@pytest.mark.parametrize('name', ['Alice', 'Bob', 'Carol-Ann', '张伟'], ids=['ascii', 'short', 'hyphen', 'unicode'])
def test_owner_name_in_statement(name):
    acc = BankAccount(name, 10.0)
    statement_text = ' '.join(acc.statement())
    assert name in statement_text

print('test_parametrize.py written.')

In [ ]:
!python -m pytest /tmp/test_parametrize.py -v 2>&1

## 6. Mocking with `unittest.mock`

Mocking replaces a real dependency (database, HTTP call, clock) with a controllable fake so tests are fast and deterministic.

In [ ]:
%%writefile /tmp/notification_service.py
import smtplib
import datetime


class NotificationService:
    """Sends email alerts when account balance drops below a threshold."""

    def __init__(self, smtp_host: str = 'mail.example.com'):
        self.smtp_host = smtp_host
        self._sent: list[dict] = []

    def send_low_balance_alert(self, account_owner: str, balance: float, threshold: float) -> bool:
        """Send alert email. Returns True if alert was sent."""
        if balance >= threshold:
            return False
        # In production this would call smtplib — we'll mock it in tests
        self._send_email(
            to=f'{account_owner.lower()}@example.com',
            subject='Low Balance Alert',
            body=f'Your balance is ${balance:.2f}, below the ${threshold:.2f} threshold.',
        )
        return True

    def _send_email(self, to: str, subject: str, body: str) -> None:
        """Low-level email dispatch (uses SMTP in production)."""
        # Simulate SMTP call
        self._sent.append({'to': to, 'subject': subject, 'body': body})

print('notification_service.py written.')

In [ ]:
%%writefile /tmp/test_mocking.py
import sys
sys.path.insert(0, '/tmp')
import pytest
from unittest.mock import MagicMock, patch, call
from notification_service import NotificationService


def test_alert_sent_when_below_threshold():
    svc = NotificationService()
    svc._send_email = MagicMock()    # replace real method with a mock

    result = svc.send_low_balance_alert('Alice', balance=45.0, threshold=100.0)

    assert result is True
    svc._send_email.assert_called_once()
    # Check the arguments passed to _send_email
    args, kwargs = svc._send_email.call_args
    assert kwargs.get('to') == 'alice@example.com' or args[0] == 'alice@example.com'


def test_no_alert_above_threshold():
    svc = NotificationService()
    svc._send_email = MagicMock()

    result = svc.send_low_balance_alert('Bob', balance=200.0, threshold=100.0)

    assert result is False
    svc._send_email.assert_not_called()


def test_alert_subject_and_body():
    svc = NotificationService()
    mock_send = MagicMock()
    svc._send_email = mock_send

    svc.send_low_balance_alert('Carol', balance=10.0, threshold=50.0)

    mock_send.assert_called_once_with(
        to='carol@example.com',
        subject='Low Balance Alert',
        body='Your balance is $10.00, below the $50.00 threshold.',
    )


@patch('notification_service.NotificationService._send_email')
def test_patch_decorator(mock_send):
    """@patch replaces the method for the duration of this test."""
    svc = NotificationService()
    svc.send_low_balance_alert('Dave', 5.0, 100.0)
    assert mock_send.called

print('test_mocking.py written.')

In [ ]:
!python -m pytest /tmp/test_mocking.py -v 2>&1

## 7. End-to-End Test Suite

Combine fixtures, parametrize, and mocking into one cohesive test file, then run it with coverage.

In [ ]:
!python -m pytest /tmp/test_bank_pytest.py /tmp/test_fixtures.py /tmp/test_parametrize.py /tmp/test_mocking.py -v --tb=short 2>&1

## Practice Exercises

**Exercise 1 — Shopping Cart**  
Write a `ShoppingCart` class with `add_item(name, price, qty)`, `remove_item(name)`, `total() -> float`, and `apply_discount(pct: float)`. Write a complete pytest test suite using at least one fixture and one `@pytest.mark.parametrize` test.

**Exercise 2 — Mock a REST Client**  
Write a `WeatherClient` class that calls `requests.get('https://api.weather.com/...')` and parses JSON. Use `unittest.mock.patch('requests.get')` to mock the HTTP response (set `.json.return_value = {...}`) and test that your parsing logic is correct without hitting the network.

**Exercise 3 — Parametrize Edge Cases**  
For a function `safe_divide(a: float, b: float) -> Optional[float]` that returns `None` for division by zero, write a parametrized test that covers: normal division, division by zero, negative numbers, very large numbers, and float precision.